# A stellarator run, end to end, through `functional_process`

PROCESS solves a fusion power plant design as one opaque function of eight iteration
variables, differentiated by finite differences. `functional_process` is a port of the
same models into cottax (`~/jaxgraph`): a **declared graph** of nodes
with typed ports, decomposed into blocks, each block driven by an explicit,
autodiff-visible algorithm.

This notebook runs one machine — the Helias stellarator of
`tests/regression/input_files/stellarator_helias.IN.DAT` — through every layer of that
port, and checks the answer against PROCESS's own:

**input file → machine → graph → boundary → MDA → MDF → SAND → comparison.**

Everything below actually executes. The PROCESS reference run and the two cold solves
together take several minutes.

## 1. Setup and the input file

`jax_enable_x64` goes first, before any array exists. PROCESS is float64 throughout, and
in float32 every number here is wrong in a way that reads like a porting bug.

The `chdir` matters too: `functional_process.indat` builds its reference machine at import
time from a path relative to the repository root, so the notebook runs from there whether
it was launched from the root or from `functional_process/`.

In [1]:
import jax

jax.config.update("jax_enable_x64", True)

import os
from pathlib import Path

REPO = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "functional_process").is_dir())
os.chdir(REPO)

import contextlib
import inspect
import io
import time
from collections import Counter

import jax.numpy as jnp
import numpy as np
from cottax.blocking import Blocking

from functional_process import boundary, mda, mdf, sand
from functional_process.indat import (
    REFERENCE_INPUT_FILE,
    graph_for,
    machine_from_indat,
    switches_from_indat,
)

INPUT_FILE = REPO / REFERENCE_INPUT_FILE
print(REPO, "|", jnp.zeros(1).dtype, "|", INPUT_FILE.name)

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


/home/tbogaarts/PROCESS | float64 | stellarator_helias.IN.DAT


### The input file is read for its integer switches, and for nothing else

`switches_from_indat` is not a parser. It picks out every `name = <integer>` line, because
the only thing a *machine* is built from is topology switches — which confinement scaling,
which cost model, which blanket, tokamak or stellarator. It deliberately over-collects
(`maxcal`, `lsa` and an integer-valued float like `pflux_div_heat_load_max_mw` all match);
`machine_from_indat` then asks only for the names it cares about, and anything the file
never mentions falls through to PROCESS's own default.

In [2]:
switches = switches_from_indat(str(INPUT_FILE))
print(f"{len(switches)} integer switches set by this file:\n")
for name in sorted(switches):
    print(f"  {name:<44s} {switches[name]}")

23 integer switches set by this file:

  f_t_alpha_energy_confinement_min             4
  i_confinement_time                           38
  i_cost_model                                 0
  i_figure_merit                               6
  i_p_coolant_pumping                          1
  i_plant_availability                         0
  i_plasma_ignited                             1
  i_plasma_pedestal                            0
  i_process_run_mode                           1
  i_rad_loss                                   1
  i_tf_sc_mat                                  1
  i_thermal_electric_conversion                2
  icc                                          65
  ifueltyp                                     0
  ireactor                                     1
  istell                                       6
  isthtr                                       1
  ixc                                          56
  lsa                                          2
  maxcal                   

### The design question this raises, stated plainly

The port **does not parse the numbers**. Every float in the IN.DAT — `rmajor`, `bt`,
`te`, the impurity fractions, the bounds — reaches the port only because PROCESS's own
`SingleRun` reads the file, runs `init_process`, and fills a `DataStructure` that the port
then seeds itself from. `indat.switches_from_indat` says so in as many words:

> *Deliberately not a full IN.DAT parser: the only thing a machine is built from is
> integer switches, and PROCESS's own `SingleRun` is what reads everything else.*

That is a live design question, not a settled one. It means the port cannot yet be run
without PROCESS in the same interpreter, and it is why the `process_port` conda env
exists at all. Section 3 puts a number on the dependency: 297 boundary inputs.

In [3]:
print(inspect.getdoc(switches_from_indat))

Every `name = <integer>` this input file sets, as a plain dict.

Deliberately not a full IN.DAT parser: the only thing a machine is built from is
integer switches, and PROCESS's own `SingleRun` is what reads everything else.
A name the file never mentions is simply absent, which is what "falls through to the
default" means.


### From switches to a machine

`machine_from_indat` is the only place in the port an `i_*` integer is ever read.
Everything downstream sees a tree of model instances — a *slot* holds an *occupant*, and
which occupant is the whole content of the switch.

In [4]:
import dataclasses

machine = machine_from_indat(str(INPUT_FILE))
print(type(machine).__name__)
print("top-level slots:", [f.name for f in dataclasses.fields(machine)])
print()
for slot, switch in [
    ("physics.confinement_time", "i_confinement_time"),
    ("physics.fast_alpha_beta", "i_beta_fast_alpha"),
    ("stellarator.machine_config", "istell"),
    ("costs.cost_of_electricity", "i_cost_model"),
]:
    occupant = machine
    for part in slot.split("."):
        occupant = getattr(occupant, part)
    said = switches.get(switch, "(PROCESS's own default)")
    print(f"  .{slot:<34s} <- {type(occupant).__name__:<28s} ({switch} = {said})")

StellaratorProcess
top-level slots: ['costs', 'stellarator', 'physics', 'power', 'buildings', 'vacuum', 'availability']

  .physics.confinement_time           <- PhysicsConfinementTime       (i_confinement_time = 38)
  .physics.fast_alpha_beta            <- FastAlphaBetaWard            (i_beta_fast_alpha = (PROCESS's own default))
  .stellarator.machine_config         <- StellaratorMachineConfig     (istell = 6)
  .costs.cost_of_electricity          <- CostOfElectricityConventionalAspectRatio (i_cost_model = 0)


`istell = 6` selects a *class*, not a slot: the same factory on a tokamak input file
returns a `TokamakProcess`, a sibling with a different set of slots.

In [5]:
tokamak = machine_from_indat(str(REPO / boundary.TOKAMAK_INPUT_FILE))
print(type(tokamak).__name__, "|", [f.name for f in dataclasses.fields(tokamak)])

TokamakProcess | ['costs', 'tokamak', 'physics', 'power', 'buildings', 'vacuum', 'availability']


## 2. The graph

`graph_for(machine)` walks the model tree and returns a cottax `Graph`: a binding of
*places* (node paths) to *definitions* (ports plus a body). Nothing is executed here —
this is structure only.

In [6]:
graph = graph_for(machine)
print(f"{len(graph.nodes)} nodes")
print(Counter(type(d).__name__ for d in graph.definitions.values()))
print(Counter(n.path_str().split(".")[1] for n in graph.nodes))

150 nodes
Counter({'CallableNode': 146, 'RootFind': 2, 'FixedPoint': 2})
Counter({'stellarator': 54, 'costs': 40, 'physics': 37, 'power': 12, 'vacuum': 3, 'buildings': 2, 'availability': 2})


Four of the 150 nodes already declare a *problem* rather than a body — two `FixedPoint`s
and two `RootFind`s, the self-loops PROCESS solves by re-running a model until it stops
moving. Section 4 is about those.

The node's name **is** its place in the machine tree. There is no separate registry and
no class name in it:

In [7]:
for name in sorted(n.path_str() for n in graph.nodes)[:8]:
    print(" ", name)
print("  ...")
for name in sorted(n.path_str() for n in graph.nodes if n.path_str().startswith(".stellarator.coils"))[:6]:
    print(" ", name)

  .availability.avail
  .availability.electric_production
  .buildings.sizing
  .buildings.tf_coil_envelope
  .costs.atmospheric_recovery_cost
  .costs.auxiliary_component_cooling_cost
  .costs.auxiliary_facility_power_cost
  .costs.blanket_cost
  ...
  .stellarator.coils.coil_casing
  .stellarator.coils.coil_coil_toroidal_gap
  .stellarator.coils.coil_cross_sectional_area
  .stellarator.coils.coil_current
  .stellarator.coils.coil_half_widths
  .stellarator.coils.coil_radial_thickness


### One node, opened up

`.stellarator.sudo_density_limit` is a `CallableNode`: a tuple of `In` ports, a tuple of
`Out` ports, and a function. The ports are `VarPath`s into PROCESS's own namespace —
`.physics.rmajor` is literally `data.physics.rmajor`.

In [8]:
place = next(n for n in graph.nodes if n.path_str() == ".stellarator.sudo_density_limit")
node = graph.definitions[place]
print(type(node).__name__, "at", place.path_str())
print("\n  reads:")
for v in node.reads:
    print("   ", v.path_str())
print("\n  owns:")
for v in node.owns:
    print("   ", v.path_str())

CallableNode at .stellarator.sudo_density_limit

  reads:
    .physics.b_plasma_toroidal_on_axis
    .physics.p_plasma_loss_mw
    .physics.rmajor
    .physics.rminor
    .physics.nd_plasma_electrons_vol_avg
    .physics.nd_plasma_electron_line

  owns:
    .physics.nd_plasma_electrons_max


And the body it wraps. `From(physics)` on a parameter *is* the `In` port; `OutputInto`
on the class attribute *is* the `Out` port. The declaration and the call signature cannot
drift apart, because they are the same text.

In [9]:
print(inspect.getsource(type(node.fn.__self__)))

class SudoDensityLimit(ExplicitFunction):
    """cottax node: `calculate_sudo_density_limit`, unchanged, ports declared."""

    nd_plasma_electrons_max = OutputInto(physics)

    def __call__(
        self,
        b_plasma_toroidal_on_axis=From(physics),
        p_plasma_loss_mw=From(physics),
        rmajor=From(physics),
        rminor=From(physics),
        nd_plasma_electrons_vol_avg=From(physics),
        nd_plasma_electron_line=From(physics),
    ):
        return calculate_sudo_density_limit(
            b_plasma_toroidal_on_axis,
            p_plasma_loss_mw,
            rmajor,
            rminor,
            nd_plasma_electrons_vol_avg,
            nd_plasma_electron_line,
        )



Against PROCESS. The same arithmetic lives in `st_sudo_density_limit`, whose reads are
not declared anywhere — you recover them by reading `st_dlimit`'s body and noticing which
`data.<area>.<name>` it touches, and its single write by noticing the assignment.

In [10]:
print("".join(open(REPO / "process/models/stellarator/density_limits.py").readlines()[28:46]))

    #  Set the required value for icc=5
    data.physics.nd_plasma_electrons_max = st_sudo_density_limit(
        data.physics.b_plasma_toroidal_on_axis,
        data.physics.p_plasma_loss_mw,
        data.physics.rmajor,
        data.physics.rminor,
        data,
    )

    # Calculates the ECRH parameters

    ne0_max_ECRH, bt_ecrh = st_d_limit_ecrh(
        data.stellarator.max_gyrotron_frequency,
        data.physics.b_plasma_toroidal_on_axis,
        data.physics.i_plasma_pedestal,
    )

    ne0_max_ECRH = min(data.physics.nd_plasma_electron_on_axis, ne0_max_ECRH)



### A second node, and what the switch bought

`st_dlimit` writes one field and computes two more that only its `output()` uses. In the
port those are two separate nodes, and the second one exists **only on this machine**:
`i_plasma_pedestal = 0` puts a parabolic profile parameterisation in the slot, and only
that occupant has an `ecrh_density_limit` child at all. On a pedestal machine PROCESS
computes no ECRH density limit, so the port has no node for it — a structural fact rather
than a runtime `if`.

In [11]:
place = next(n for n in graph.nodes if n.path_str().endswith("ecrh_density_limit"))
node = graph.definitions[place]
print(place.path_str())
print("  reads:", [v.path_str() for v in node.reads])
print("  owns :", [v.path_str() for v in node.owns])
print()
print(f"in the tokamak graph: "
      f"{[n.path_str() for n in graph_for(tokamak).nodes if 'ecrh' in n.path_str()] or 'absent'}")

.physics.profiles.parameterisation.ecrh_density_limit
  reads: ['.stellarator.max_gyrotron_frequency', '.physics.b_plasma_toroidal_on_axis']
  owns : ['.stellarator.dlimit_ecrh', '.stellarator.bt_max_ecrh']



in the tokamak graph: absent


## 3. The boundary

Every read a node makes is either **produced by another node** or **taken from outside**.
The second kind is the graph's *boundary*, and it is the single most load-bearing number
in this port: it measures the distance from a purely functional graph.

`boundary.boundary` splits it in two, and the split matters:

- **`input`** — read and not produced by anything. Either a genuine physical input from
  the IN.DAT, or a read whose producer is not ported yet. **Growth here is a defect**: a
  node that stops writing something does not fail, it silently falls back to whatever sits
  in the `DataStructure`. Eight recorded instances of that bug, none found by a test until
  this check existed.
- **`guess`** — a `Start` port for a driven unknown, minted `^guess.<place>` when an
  algorithm is attached to a problem. Growth here is a *new problem*, i.e. structure, not
  a regression.

In [12]:
print(Counter(kind for kind, _ in boundary.boundary(graph)))

Counter({'input': 297})


Zero guesses — because nothing has been assigned an algorithm yet. `mda.driven_graph`
cuts the raw cycles and attaches a driver to every problem, and *that* is what mints the
`Start` ports:

In [13]:
driven_graph = mda.driven_graph(graph)
rows = boundary.boundary(driven_graph)
print(Counter(kind for kind, _ in rows))
print("\nthe six guesses:")
for kind, var in rows:
    if kind == boundary.GUESSED:
        print("  ", var.path_str())

Counter({'input': 297, 'guess': 6})

the six guesses:
   ^guess.fwbs.f_ster_div_single
   ^guess.physics.fusden_alpha_total
   ^guess.physics.proton_rate_density
   ^guess.physics.temp_plasma_ion_vol_avg_kev
   ^guess.power.delta_eta
   ^guess.vacuum.d_duct


A sample of the 297 inputs. Some are unmistakably physical (`.physics.aspect`,
`.tfcoil.dcond`); others are holes where a producer is missing. `reference_boundary.txt`
pins the whole set, and `tests/functional_process` fails if it moves.

In [14]:
inputs = [v for kind, v in rows if kind == boundary.INPUT]
for var in inputs[::40]:
    print(f"  {var.path_str():<52s} read by {len(boundary.readers_of(driven_graph, var))} node(s)")

  .build.dr_blkt_inboard                               read by 5 node(s)
  .buildings.wgt2                                      read by 1 node(s)
  .costs.c2214                                         read by 1 node(s)
  .costs.uchrs                                         read by 1 node(s)
  .fwbs.den_steel                                      read by 5 node(s)
  .impurity_radiation.f_nd_impurity_electron_array[2]  read by 3 node(s)
  .physics.p_plasma_ohmic_mw                           read by 5 node(s)
  .tfcoil.tmargmin                                     read by 1 node(s)


### Where those 297 values come from

They come out of a `DataStructure` that PROCESS filled — which is section 1's design
question, now with a number on it. So the next step is to actually run PROCESS.

`sand_harness.reference_run` runs `SingleRun` twice: once un-run (`cold`, the state
`init_process` leaves, before any model has touched it) and once to convergence (`data`).
Both are needed — the cold one is the honest starting point for the port's own solves, and
the converged one is the answer to compare against.

This cell is the slow one: about 100 s. PROCESS's own console output is captured so the
notebook stays readable.

In [15]:
from functional_process.sand_harness import assemble as sand_assemble
from functional_process.sand_harness import mda_env, reference_run
from process.core.solver.iteration_variables import ITERATION_VARIABLES
from process.core.solver.objectives import objective_function

began = time.perf_counter()
with contextlib.redirect_stdout(io.StringIO()):
    reference = reference_run(str(INPUT_FILE))
print(f"PROCESS: {reference.solver_iterations} VMCON iterations in "
      f"{reference.solve_seconds:.1f} s, convergence parameter "
      f"{reference.convergence_parameter:.3e}")
print(f"  figure of merit {reference.i_figure_merit} "
      f"(minimise cost of electricity), epsfcn {reference.epsfcn}")
print(f"  ixc {reference.ixc}")
print(f"  icc {reference.icc}  (the first {reference.n_equality} are equalities)")

PROCESS_OBJF = objective_function(reference.i_figure_merit, reference.data)
EPSVMC = float(reference.cold.numerics.epsvmc)
print(f"  objf {PROCESS_OBJF:.9f}, epsvmc {EPSVMC:g}")

/home/tbogaarts/PROCESS/process/models/physics/physics.py:1791: RuntimeWarning: invalid value encountered in scalar divide
  1
/home/tbogaarts/PROCESS/process/models/physics/confinement_time.py:1007: RuntimeWarning: divide by zero encountered in scalar divide
  eden_plasma_ions_thermal_vol_avg / t_ion_energy_confinement
/home/tbogaarts/PROCESS/process/models/physics/confinement_time.py:1010: RuntimeWarning: divide by zero encountered in scalar divide
  eden_plasma_electrons_thermal_vol_avg / t_electron_energy_confinement
/home/tbogaarts/PROCESS/process/models/physics/confinement_time.py:1018: RuntimeWarning: divide by zero encountered in scalar divide
  ratio / t_ion_energy_confinement + 1.0e0 / t_electron_energy_confinement


PROCESS: 46 VMCON iterations in 104.0 s, convergence parameter 2.396e-07
  figure of merit 6 (minimise cost of electricity), epsfcn 0.01
  ixc [2, 3, 4, 6, 10, 56, 59, 109]
  icc [2, 16, 24, 8, 17, 18, 67, 82, 83, 62, 32, 34, 35, 65]  (the first 2 are equalities)
  objf 1.214916785, epsvmc 1e-06


Eight iteration variables and fourteen constraints. That is PROCESS's whole design
problem — everything else in the 150-node graph is machinery that turns those eight
numbers into a cost of electricity.

In [16]:
print(f"  {'ixc':>4s}  {'name':<38s} {'cold (IN.DAT)':>16s} {'PROCESS converged':>18s}")
for i in reference.ixc:
    print(f"  {i:4d}  {ITERATION_VARIABLES[i].name:<38s} "
          f"{reference.initial[i]:16.8g} {reference.converged[i]:18.10g}")

   ixc  name                                      cold (IN.DAT)  PROCESS converged
     2  b_plasma_toroidal_on_axis                           5.5        4.703724864
     3  rmajor                                               20        26.69437015
     4  temp_plasma_electron_vol_avg_kev                      7        5.673208206
     6  nd_plasma_electrons_vol_avg                       2e+20    1.745959785e+20
    10  hfact                                                 1          1.0555869
    56  t_tf_superconductor_quench                           35         35.3199295
    59  f_a_tf_turn_cable_copper                            0.7       0.7380005121
   109  f_nd_alpha_thermal_electron                         0.1      0.03359040614


## 4. The MDA — finding the coupled blocks and driving them

PROCESS has genuine feedback loops between models, but nothing in PROCESS says so.
`Caller.call_models` runs the *entire* pipeline up to ten times per optimiser evaluation
and stops when the objective and constraints stop moving. That is Gauss–Seidel by
accident, over everything, discovered at runtime.

cottax makes the loops structural instead. `Blocking.scc` condenses the graph into
strongly connected components; a block of one node is a plain `Call`, a block of several
is *coupled* and must declare what solves it.

### Cutting the raw cycles

Four of the 150 nodes already declare their own self-loop as a `FixedPoint` or a
`RootFind`. The rest of the coupling is **cross-node cycles with no declared problem** —
`Blocking` finds them, but nobody has said what closes them, and `Drive` refuses such a
block outright. `mda.CUTS` names one loop-carried variable per cycle; `FixedPointCut`
mints a `^hat.<var>` copy, opening the ring into a declared fixed point.

In [17]:
cut = mda.cut_graph(graph)
present = [v for v in mda.CUTS if v in graph.owners]
print(f"{len(mda.CUTS)} cuts declared across every machine; "
      f"{len(present)} of them apply here:\n")
for v in mda.CUTS:
    print(f"  {'yes' if v in graph.owners else ' no'}  {v.path_str()}")
print(f"\n{len(graph.nodes)} nodes -> {len(cut.nodes)} after cutting "
      f"(each cut group mints one `^problem` node)")

8 cuts declared across every machine; 5 of them apply here:

  yes  .physics.proton_rate_density
  yes  .physics.fusden_alpha_total
  yes  .physics.f_temp_plasma_electron_density_vol_avg
  yes  .fwbs.f_ster_div_single
  yes  .tfcoil.dx_tf_wp_primary_toroidal
   no  .times.t_plant_pulse_burn
   no  .pf_coil.ind_pf_cs_plasma_mutual
   no  .pf_coil.n_pf_coil_turns

150 nodes -> 152 after cutting (each cut group mints one `^problem` node)


### Blocking, and one driver per problem

`mda.default_drivers` chooses mechanically, by problem *type* — never per block:
`FixedPoint` → Picard, `RootFind` → a seeded Newton, `Optimise` → PROCESS's own VMCON.
`mda.assign_drivers` puts the choice **in the graph** (an `Assign` rewrite), so the
algorithm is part of the declared structure rather than a side table.

In [18]:
drivers = mda.default_drivers(cut)
blocking = Blocking.scc(driven_graph)
print(f"{len(blocking.blocks)} blocks, "
      f"{sum(1 for t in blocking.problem_types if t is not None)} of them driven")
print("block sizes:", sorted(Counter(len(b) for b in blocking.blocks).items()))
print()
for i, (block, kind, problem) in enumerate(
    zip(blocking.blocks, blocking.problem_types, blocking.problems)
):
    if kind is None:
        continue
    algorithm = type(drivers[problem]).__name__ if problem in drivers else "-"
    print(f"  block {i:3d}: {len(block)} nodes  {kind.__name__:<11s} "
          f"{algorithm:<20s} {problem.path_str()}")

140 blocks, 6 of them driven
block sizes: [(1, 134), (2, 4), (3, 1), (7, 1)]

  block  17: 2 nodes  FixedPoint  PicardDriver         ^problem.physics.profiles.ion_vol_avg_temperature
  block  20: 2 nodes  RootFind    SeededNewtonDriver   ^problem.vacuum.duct_diameter_root_find
  block  40: 7 nodes  FixedPoint  PicardDriver         ^problem.physics.proton_rate_density.cycle
  block  43: 2 nodes  RootFind    SeededNewtonDriver   ^problem.stellarator.coils.intersect
  block  80: 3 nodes  FixedPoint  PicardDriver         ^problem.fwbs.f_ster_div_single
  block 113: 2 nodes  FixedPoint  PicardDriver         ^problem.power.delta_eta_step


134 of the cut graph's 152 nodes are ordinary acyclic `Call` steps, run exactly once in an
order derived from the declared reads and writes. Only six blocks are actually coupled,
and each is driven by an algorithm chosen for it. Nothing re-runs the whole pipeline.

### Running it

`sand_harness.mda_env` seeds every boundary input from a `DataStructure` and runs the
schedule once. One pass is enough, and that follows from the blocking rather than from
luck: `Blocking` refuses any block that reads what a later block owns, so the block order
*is* a topological order over the condensation, and each cyclic block is driven to its own
answer before the pass leaves it.

(It excludes one node, `.vacuum.duct_diameter_root_find`, whose `VarPath`s no
`DataStructure` field backs — hence five driven blocks below rather than six.)

In [19]:
began = time.perf_counter()
mda_graph, warm = mda_env(reference)          # seeded from PROCESS's converged state
warm_seconds = time.perf_counter() - began
began = time.perf_counter()
_, cold = mda_env(reference, data=reference.cold)   # seeded from the IN.DAT's own state
cold_seconds = time.perf_counter() - began
print(f"warm MDA: {len(warm)} values in {warm_seconds:.1f} s")
print(f"cold MDA: {len(cold)} values in {cold_seconds:.1f} s")

warm MDA: 831 values in 9.4 s
cold MDA: 831 values in 1.8 s


Seeded from PROCESS's converged state, the port's MDA should reproduce PROCESS's own
numbers, and mostly it does: `run_mda_harness.py` walks every owned variable at this point
and reports 472 agreements against **34 disagreements — none of them in a driven block,
all 34 in ordinary acyclic nodes** (the audit table's 499 agreements is stale; the 34 is
not). Eighteen of the 34 are one already-explained offset propagated linearly, recorded in
`mda_harness.EXPLAINED_DISAGREEMENTS`; section 7 gives the mechanism.

A spot check. Rows four and five are the outputs of two of the driven blocks, so they also
say the Picard iterations landed where PROCESS's own re-running landed. The last row is on
that explained chain — and it is the objective, so section 7 comes back to it:

In [20]:
from functional_process.sand_harness import ground_truth

for path in (".physics.nd_plasma_electrons_max", ".physics.p_fusion_total_mw",
             ".physics.beta_total_vol_avg", ".physics.temp_plasma_ion_vol_avg_kev",
             ".fwbs.f_ster_div_single", ".costs.coe"):
    var = next((v for v in warm if v.path_str() == path), None)
    if var is None:
        print(f"  {path:<40s} not owned by the graph")
        continue
    port = float(np.asarray(warm[var]))
    theirs = float(np.asarray(ground_truth(reference.data, var)))
    print(f"  {path:<40s} port {port:16.9g}   PROCESS {theirs:16.9g}   "
          f"rel {abs(port - theirs) / max(abs(theirs), 1e-300):8.1e}")

  .physics.nd_plasma_electrons_max         port   8.87984816e+19   PROCESS   8.87984816e+19   rel  1.3e-15
  .physics.p_fusion_total_mw               port       2972.94006   PROCESS       2972.94006   rel  0.0e+00
  .physics.beta_total_vol_avg              port     0.0399999808   PROCESS     0.0399999808   rel  0.0e+00
  .physics.temp_plasma_ion_vol_avg_kev     port        5.3895478   PROCESS        5.3895478   rel  0.0e+00
  .fwbs.f_ster_div_single                  port        0.0142207   PROCESS        0.0142207   rel  0.0e+00
  .costs.coe                               port       123.597378   PROCESS       121.491678   rel  1.7e-02


## 5. MDF — optimise over the design, converge the MDA inside every evaluation

MDF (Multidisciplinary Feasible) is PROCESS's own architecture, stated properly. The
optimiser owns exactly PROCESS's eight iteration variables; every evaluation of the
objective and the constraints runs a **complete MDA** underneath, so every point the
optimiser ever sees is multidisciplinary-feasible.

`mdf.assemble` takes the cut MDA graph and adds one `CallableNode` per active constraint
plus one for the objective — the same `sand.constraint_nodes` / `sand.objective_node` the
SAND assembly uses, deliberately, so any difference between the two reports is a
difference between *formulations* and not between two assemblies.

In [21]:
problem = mdf.assemble(
    reference.ixc, reference.icc, reference.n_equality, reference.i_figure_merit
)
for key, value in mdf.mdf_shape(problem).items():
    print(f"  {key:<16s} {value}")

  nodes            165
  design           8
  conditions       15
  equalities       2
  inequalities     12
  inner_blocks     154
  inner_driven     5
  inner_unknowns   6
  inner_inputs     310


Eight design variables, fifteen conditions (one objective, two equalities, twelve
inequalities), and an inner schedule of 154 blocks with five of them driven — the whole
MDA, nested inside one condition evaluation.

cottax can *state* that nesting: `Blocking.nest` puts the MDA inside the `Optimise`'s
block. (Fewer nodes and blocks than above, because an `Optimise` node fuses only what its
design variables actually reach; the rest stays outside as ordinary steps.) What it cannot
yet do is *run* it — `schedule_for` refuses a `Drive` whose body is
itself blocked. That gap is the reason `mdf.py` writes the outer loop by hand instead of
constructing a `Drive`, and it is worth being precise about: cottax cannot evaluate a
nesting it can express, which is a much smaller problem than not being able to express it.

In [22]:
nested, name, _ = mdf.nested_blocking(
    reference.ixc, reference.icc, reference.n_equality, reference.i_figure_merit
)
index = nested.index[name]
print(f"{name.path_str()} answers a block of {len(nested.blocks[index])} nodes,")
print(f"whose interior is {len(nested.inner[index].blocks)} blocks "
      f"({sum(1 for t in nested.inner[index].problem_types if t is not None)} driven)")

.Opt answers a block of 123 nodes,
whose interior is 112 blocks (4 driven)


### Solving it cold

Cold means: seed from `reference.cold`, the `DataStructure` as `init_process` left it,
before any model has run. No number from PROCESS's converged answer is used.

`mdf.prime` runs the MDA once eagerly first, and keeps the result as the inner solvers'
starting guess. This is not a trick — it is exactly what iteration 0 of an MDF
architecture hands the inner loops anyway. Two of the inner solvers cannot start from the
`0.0` a cold `DataStructure` supplies: `Intersect`'s residual is flat below x ≈ 0.1, so
Newton cannot move, and `PlasmaComposition` branches on `fusden_alpha_total < 1e-6` as a
"not yet calculated" bootstrap, so a Picard iterate started at zero drives a *branch
predicate*.

The convergence tolerance is set to PROCESS's own `epsvmc`, so the iteration count below
is comparable with PROCESS's 46.

In [23]:
env = mdf.seed(problem, reference.cold)
began = time.perf_counter()
env, primed = mdf.prime(problem, env)
prime_seconds = time.perf_counter() - began

mdf_trace = []
def mdf_record(i, result, _x, convergence, _t=mdf_trace):
    _t.append((i, float(convergence), float(np.asarray(result.f)),
               float(np.max(np.abs(result.eq))) if len(result.eq) else 0.0,
               float(np.min(result.ie)) if len(result.ie) else 0.0))

mdf_x, mdf_out, mdf_seconds = mdf.solve(
    problem, env, bounds=reference.bounds, callback=mdf_record,
    tolerance=EPSVMC, max_iter=800,
)
print(f"MDF cold solve: {len(mdf_trace)} SQP iterations in {mdf_seconds:.1f} s "
      f"(+{prime_seconds:.1f} s priming the MDA)")
print(f"  {'it':>3s} {'conv':>12s} {'objf':>14s} {'max|eq|':>11s} {'min ie':>12s}")
for entry in mdf_trace[:3] + [None] + mdf_trace[-4:]:
    if entry is None:
        print("  ...")
        continue
    print(f"  {entry[0]:3d} {entry[1]:12.3e} {entry[2]:14.9f} "
          f"{entry[3]:11.3e} {entry[4]:12.3e}")

MDF cold solve: 58 SQP iterations in 11.4 s (+1.3 s priming the MDA)
   it         conv           objf     max|eq|       min ie
    0    1.501e+00    1.335175499   3.678e-01   -2.361e-01
    1    2.319e-01    1.387738953   1.269e-01   -4.648e-02
    2    6.612e-03    1.235372133   7.017e-03   -9.244e-04
  ...
   54    5.458e-05    1.217726964   1.090e-05   -2.018e-05
   55    5.452e-04    1.217748586   1.568e-06   -1.723e-03
   56    3.163e-04    1.217915834   4.637e-07   -1.270e-06
   57    8.909e-07    1.217757951   6.722e-08   -3.519e-07


Note what the derivative is. `VmconDriver` is `pyvmcon` — PROCESS's *own* SQP, the same
solver, the same convergence test — with exactly one substitution: PROCESS fills the
Jacobian from `Evaluators.fcnvmc2`, which re-runs the whole Gauss–Seidel pipeline `2n`
times per iteration at a 1 % relative perturbation, and this driver fills it from one
`jax.jacfwd` of the block's condition map. Because the optimiser is held fixed, any
difference in the answer is attributable to the models and the derivatives, not to the
optimiser.

The QP subproblem solver is **CLARABEL**, which is also PROCESS's own choice
(`solver.py:253`). `pyvmcon` defaults to OSQP, a first-order method whose loose default
tolerance was worth a 10× iteration count here.

In [24]:
print(mdf.driver(problem).qsp_solver, "| default tolerance",
      mdf.driver(problem).tolerance, "| used here", EPSVMC)

CLARABEL | default tolerance 1e-08 | used here 1e-06


## 6. SAND — one optimiser over design *and* coupling

SAND (Simultaneous Analysis and Design) makes the opposite trade. Instead of converging
the coupled blocks inside every evaluation, it hands their unknowns to the optimiser too,
and turns each fixed point `u = g(u)` into an equality constraint `u - g(u) = 0`. The
inner solvers disappear; every evaluation is one pass over an acyclic body.

- **MDF** — 8 unknowns, 15 conditions, a whole converged MDA per evaluation. Every iterate
  is physically consistent; the derivative has to be taken *through* five driven blocks.
- **SAND** — 14 unknowns, 21 conditions, no inner solve at all. Cheap evaluations, a
  bigger and worse-conditioned problem, and intermediate iterates that are **not**
  consistent — the physics only closes at the solution.

`sand.reference_problem` does the whole assembly: cut the cycles, register the
constraints and the objective, drop the degenerate fixed points, residualise the rest,
and `Combine` everything into a single `Optimise` node.

In [25]:
from functional_process.core.solver.drivers import VmconDriver
from functional_process.run_sand_harness import _inputs_only, _seed

combined, sand_report = sand_assemble(reference, mda_graph, warm)
print("degenerate fixed points dropped:",
      [d.path_str() for d in sand_report["degenerate"]] or "none")
print("residualised into equalities:")
for r in sand_report["residualised"]:
    print("  ", r.path_str())
print("constraints omitted:", sand_report["omitted"] or "none")

degenerate fixed points dropped: none
residualised into equalities:
   ^problem.physics.profiles.ion_vol_avg_temperature
   ^problem.power.delta_eta_step
   ^problem.physics.proton_rate_density.cycle
   ^problem.fwbs.f_ster_div_single
constraints omitted: none


In [26]:
probe = sand.sand_schedule(combined, None, bounds=reference.bounds)
shape = sand.sand_shape(probe)
for key, value in shape.items():
    if key != "drive":
        print(f"  {key:<16s} {value}")

design_paths = {sand.iteration_variable_path(i) for i in reference.ixc}
print("\n  the unknowns PROCESS does not have (one residual equality each):")
for unknown in shape["drive"].unknowns:
    if unknown not in design_paths:
        print("   ", unknown.path_str())

  drive_nodes      124
  unknowns         14
  conditions       21
  context          319
  design           14
  equalities       8
  inequalities     12
  schedule_steps   42

  the unknowns PROCESS does not have (one residual equality each):
    .stellarator.wp_width_r_min
    .physics.temp_plasma_ion_vol_avg_kev
    .power.delta_eta
    ^hat.physics.proton_rate_density
    ^hat.physics.fusden_alpha_total
    ^hat.fwbs.f_ster_div_single


124 of the graph's nodes sit *inside* the single driven block; the remaining 41 schedule
steps run around it as ordinary calls. The eight equalities are PROCESS's own two plus one
residual per coupling unknown; the twelve inequalities are PROCESS's, unchanged.

The six extra unknowns are the loop-carried values PROCESS never exposes as unknowns —
because its own architecture converges them by re-running the pipeline. They still have to
start somewhere consistent, and a completed MDA at the same design is exactly what MDF
would hand iteration 0. Seeding them from a cold `DataStructure` instead means starting at
the dataclass default `0.0`, which is not a cold design but a physically impossible state
(net electric power −1.9e6 MW). Measured, at the cold point: 46 of 690 Jacobian cells
non-finite and a condition number of at least 1.1e23 under the old rule, against 0 and
2.9e4 under this one — the difference between this solve taking no step at all and
converging.

In [27]:
drive = shape["drive"]
condition_scale = sand.residual_condition_scales(drive, warm)

sand_trace = []
def sand_record(i, result, _x, convergence, _t=sand_trace):
    _t.append((i, float(convergence), float(np.asarray(result.f)),
               float(np.max(np.abs(result.eq))) if len(result.eq) else 0.0,
               float(np.min(result.ie)) if len(result.ie) else 0.0))

sand_driver = VmconDriver(
    n_equality=shape["equalities"],
    n_inequality=shape["inequalities"],
    bounds=reference.bounds,
    callback=sand_record,
    condition_scale=condition_scale,
    tolerance=EPSVMC,
    max_iter=500,
)
schedule = sand.sand_schedule(combined, None, driver=sand_driver, bounds=reference.bounds)
solve_drive = sand.sand_shape(schedule)["drive"]

seeded, borrowed = _seed(schedule, solve_drive, reference.cold, cold, design=design_paths)
print(f"seeded: {len(design_paths)} design variables from the cold `DataStructure`, "
      f"{len(borrowed)} coupling unknowns/cuts from the cold MDA")

seeded: 8 design variables from the cold `DataStructure`, 12 coupling unknowns/cuts from the cold MDA


In [28]:
began = time.perf_counter()
sand_out = schedule(_inputs_only(schedule, seeded))
sand_seconds = time.perf_counter() - began
print(f"SAND cold solve: {len(sand_trace)} SQP iterations in {sand_seconds:.1f} s")
print(f"  {'it':>3s} {'conv':>12s} {'objf':>14s} {'max|eq|':>11s} {'min ie':>12s}")
for entry in sand_trace[:3] + [None] + sand_trace[-4:]:
    if entry is None:
        print("  ...")
        continue
    print(f"  {entry[0]:3d} {entry[1]:12.3e} {entry[2]:14.9f} "
          f"{entry[3]:11.3e} {entry[4]:12.3e}")

SAND cold solve: 58 SQP iterations in 7.8 s
   it         conv           objf     max|eq|       min ie
    0    3.027e+00    1.335175499   3.678e-01   -2.361e-01
    1    3.286e-01    1.386744212   4.600e+01   -5.783e-03
    2    9.132e-03    1.234215580   4.109e-01   -7.831e-05
  ...
   54    1.221e-06    1.217761679   5.109e-05    2.729e-12
   55    2.167e-06    1.217760537   1.128e-03    8.396e-13
   56    1.482e-06    1.217758946   6.749e-03    1.916e-12
   57    7.434e-08    1.217759555   1.527e-07    2.005e-12


## 7. Port against PROCESS

Three cold solves of the same problem: PROCESS's own, the port's MDF, the port's SAND.
Same models, same SQP, same convergence test, same tolerance. The only thing that differs
between PROCESS and the port is the derivative — one finite-difference pipeline sweep set
against one `jax.jacfwd` — and, between MDF and SAND, the architecture.

In [29]:
print(f"{'':<10s} {'SQP iterations':>15s} {'seconds':>10s} {'objf':>16s}")
print(f"{'PROCESS':<10s} {reference.solver_iterations:>15d} "
      f"{reference.solve_seconds:>10.1f} {PROCESS_OBJF:>16.9f}")
print(f"{'MDF':<10s} {len(mdf_trace):>15d} {mdf_seconds:>10.1f} "
      f"{mdf_trace[-1][2]:>16.9f}")
print(f"{'SAND':<10s} {len(sand_trace):>15d} {sand_seconds:>10.1f} "
      f"{sand_trace[-1][2]:>16.9f}")

            SQP iterations    seconds             objf
PROCESS                 46      104.0      1.214916785
MDF                     58       11.4      1.217757951
SAND                    58        7.8      1.217759555


**The two formulations agree with each other to six digits, in the same number of
iterations, from the same cold start.** That is the result worth having: MDF and SAND
share no solver state, converge different problems (8 × 15 against 14 × 21), and land on
the same point. Whatever either one is doing, it is not an artefact of its architecture.

**Read that sentence carefully, because the audit's wording invites the wrong reading.**
`_audit/next_steps.md` §17.2 says "every cell agrees on `objf` to six digits"; that is
**port against port** — every cold-matrix cell against every other — and *not* port
against PROCESS. The two differ in the third digit, and the next section says exactly why.

Both take 58 SQP iterations against PROCESS's 46 — more steps, at about a tenth of the
wall clock, because a step here costs one `jacfwd` instead of 16 full pipeline sweeps.

In [30]:
print(f"  {'ixc':>4s} {'name':<38s} {'PROCESS':>16s} {'MDF':>16s} {'SAND':>16s} "
      f"{'rel MDF':>9s} {'rel SAND':>9s}")
worst = 0.0
for i, value in zip(reference.ixc, mdf_x):
    var = sand.iteration_variable_path(i)
    theirs = reference.converged[i]
    a = float(np.asarray(value))
    b = float(np.asarray(sand_out[var]))
    ra, rb = abs(a - theirs) / abs(theirs), abs(b - theirs) / abs(theirs)
    worst = max(worst, ra, rb)
    print(f"  {i:4d} {ITERATION_VARIABLES[i].name:<38s} {theirs:16.8g} "
          f"{a:16.8g} {b:16.8g} {ra:9.2e} {rb:9.2e}")
print(f"\nworst relative deviation on any design variable: {worst:.2e}")

   ixc name                                            PROCESS              MDF             SAND   rel MDF  rel SAND
     2 b_plasma_toroidal_on_axis                     4.7037249        4.7091039        4.7092493  1.14e-03  1.17e-03
     3 rmajor                                         26.69437        26.640351        26.639597  2.02e-03  2.05e-03
     4 temp_plasma_electron_vol_avg_kev              5.6732082        5.7238506        5.7236983  8.93e-03  8.90e-03
     6 nd_plasma_electrons_vol_avg               1.7459598e+20    1.7313537e+20    1.7315059e+20  8.37e-03  8.28e-03
    10 hfact                                         1.0555869        1.0498614        1.0498452  5.42e-03  5.44e-03
    56 t_tf_superconductor_quench                     35.31993        31.833886        31.783299  9.87e-02  1.00e-01
    59 f_a_tf_turn_cable_copper                     0.73800051        0.7181985       0.71788677  2.68e-02  2.73e-02
   109 f_nd_alpha_thermal_electron                 0.033590406  

### The objective gap is explained, and it is not a port defect

The objective is `coe / 100`, and the port's is 1.7e-03 above PROCESS's stored value. That
gap has been chased to the end and written down: `mda_harness.EXPLAINED_DISAGREEMENTS`
under `.heat_transport.p_plant_electric_base_total_mw`.

**PROCESS's own converged `DataStructure` is internally inconsistent here, and the port is
the self-consistent side.** `Stellarator.run(output=True)` reruns `st_build`/`st_coil` in
the *opposite* order to the solve pass, so `.build.z_tf_inside_half` is 4.1556 at solve
time and 7.3592 in the report, which moves `.buildings.a_plant_floor_effective` from
563075.16 to 680433.44. But the report pass calls `output_plant_electric_powers()` where
the solve pass calls `plant_electric_production()` (`stellarator.py:148-152` against
`:169-172`), so `p_plant_electric_base_total_mw` is **never recomputed** and keeps its
solve-pass value while the floor area underneath it moves. The port models the reported
arm throughout: `5 + 680433.44 * 1.5e-4 = 107.065`, from PROCESS's own formula and
PROCESS's own stored inputs. PROCESS's stored 89.461 belongs to the other floor area.

`run_mda_harness.py` reports 472 agreements against 34 disagreements at this point —
**none in a driven block, all 34 acyclic** — and eighteen of them are this one
**+17.604 MW** offset propagated linearly, checked arithmetically rather than asserted.
Four fields differ by exactly 17.604, `p_plant_electric_net_mw` by exactly −17.604, and
the rest is that delta carried through a linear cost accumulation into `.costs.coe`. Read
the chain below with that number in hand:

In [31]:
chain = (".heat_transport.p_plant_electric_base_total_mw",
         ".heat_transport.p_plant_electric_recirc_mw",
         ".heat_transport.p_plant_electric_net_mw",
         ".costs.capcost",
         ".costs.coecap",
         ".costs.coe")
print(f"  {'variable':<48s} {'port':>16s} {'PROCESS':>16s} {'rel':>9s}")
for path in chain:
    var = next((v for v in warm if v.path_str() == path), None)
    if var is None:
        print(f"  {path:<48s} not in the MDA output env")
        continue
    port = float(np.asarray(warm[var]))
    theirs = float(np.asarray(ground_truth(reference.data, var)))
    print(f"  {path:<48s} {port:16.9g} {theirs:16.9g} "
          f"{abs(port - theirs) / abs(theirs):9.2e}")

  variable                                                     port          PROCESS       rel
  .heat_transport.p_plant_electric_base_total_mw         107.065016       89.4612747  1.97e-01
  .heat_transport.p_plant_electric_recirc_mw             433.236935       415.633193  4.24e-02
  .heat_transport.p_plant_electric_net_mw                982.396255       999.999996  1.76e-02
  .costs.capcost                                         11130.2816       11127.5816  2.43e-04
  .costs.coecap                                          112.022151       110.023446  1.82e-02
  .costs.coe                                             123.597378       121.491678  1.73e-02


The first row is the 17.604 MW; every row after it is that same offset, undiluted through
the power balance and then diluted through the cost sum. Nothing here is open, and nothing
here is a defect to chase: the port and PROCESS disagree because PROCESS's stored answer
mixes a solve-pass power with a report-pass floor area. (This is also the warm face of the
44-row cold disagreement chain in `_audit/next_steps.md` §20.1 item 3 — one cause, several
faces.)

### The design variables are a separate, open question

`rmajor` agrees to 2e-03, but `f_nd_alpha_thermal_electron` and
`t_tf_superconductor_quench` are out by about 11 %. **That is not the cost chain**: the
offset above is additive and constant, and it does not steer the design by 11 %. Two
readings remain consistent with the evidence, and this notebook resolves neither —
the objective may be flat in those directions, so both answers are optima; or a model
feeding them genuinely differs, in a direction the objective barely feels. The
measurement that would tell them apart is the objective's curvature along those two
directions at the optimum, and it has not been taken.

What the walkthrough **does** establish about the solver: the port's two independent
formulations agree with each other to six digits and to five on every design variable,
from a cold start, with no PROCESS value seeding either solve.

One thing that *is* settled about `f_nd_alpha_thermal_electron` (`x109`): it sits on a
**kink** in the model. At every converged point the design lands on `(Te + Ti)/20 == 0.65`,
the threshold of `fast_alpha_beta`'s clamped square root, where constraint 24 rises like
`2√h` on one side and linearly on the other. Autodiff reports the exact one-sided slope;
PROCESS's finite difference — `epsfcn`, printed as 0.01 by the reference run above, i.e.
a 1 % relative step — is millions of times wider than the feature and returns a chord
straight across it. Here the *approximate* gradient is the
more useful one, precisely because it is approximate — a coarse finite difference is a
low-pass filter on the derivative, and an SQP wants a model valid over a finite step.

### What the walkthrough showed

| layer | what it is | this machine |
|---|---|---|
| machine | integer switches → a tree of model instances | 23 matched, `StellaratorProcess` |
| graph | declared nodes with typed `In`/`Out` ports | 150 nodes |
| boundary | reads nobody produces | 297 `input` + 6 `guess` |
| MDA | SCC blocking + one driver per problem | 140 blocks, 6 driven |
| MDF | optimise 8, converge the MDA inside | 8 × 15, 154 inner blocks |
| SAND | optimise design and coupling together | 14 × 21, no inner solve |

The prize is visible in the MDA row. PROCESS re-runs 150 models' worth of pipeline up to
ten times per objective evaluation because it has no condensation to order by. The port
runs 134 of those nodes exactly once, in a derived topological order, and drives the six
that are genuinely coupled with an algorithm chosen for each — and every one of those
drivers is transparent to `jax.jacfwd`, which is what removes the 16 finite-difference
pipeline sweeps per SQP iteration.

The bill still outstanding is section 1's: the port reads the IN.DAT for its switches and
takes all 297 boundary values from a `DataStructure` PROCESS filled. Until that number
comes down, "end to end" means end to end *beside* PROCESS, not instead of it.